generate-summary-json.py
========================

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import json
from typing import List, Dict, Optional, Tuple
import glob
import os

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import json
from typing import Dict, List, Tuple, Optional
import glob
import os

class MAFAnalyzer:
    def __init__(self):
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s'
        )
        self.logger = logging.getLogger(__name__)
        
        # Define comprehensive column sets
        self.variant_info_columns = [
            'Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build',
            'Chromosome', 'Start_Position', 'End_Position', 'Strand',
            'Variant_Classification', 'Variant_Type', 'Reference_Allele',
            'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'dbSNP_RS',
            'dbSNP_Val_Status', 'Tumor_Sample_Barcode', 'Matched_Norm_Sample_Barcode'
        ]
        
        self.sequencing_info_columns = [
            'Match_Norm_Seq_Allele1', 'Match_Norm_Seq_Allele2',
            'Tumor_Validation_Allele1', 'Tumor_Validation_Allele2',
            'Match_Norm_Validation_Allele1', 'Match_Norm_Validation_Allele2',
            'Verification_Status', 'Validation_Status', 'Mutation_Status',
            'Sequencing_Phase', 'Sequence_Source', 'Validation_Method',
            'Score', 'BAM_File', 'Sequencer'
        ]
        
        self.consequence_columns = [
            'HGVSc', 'HGVSp', 'HGVSp_Short', 'Transcript_ID',
            'Exon_Number', 'all_effects', 'Consequence', 'IMPACT',
            'SIFT', 'PolyPhen', 'CLIN_SIG'
        ]
        
        self.coverage_columns = [
            't_depth', 't_ref_count', 't_alt_count',
            'n_depth', 'n_ref_count', 'n_alt_count'
        ]

    def setup_directories(self) -> Tuple[Path, Path]:
        """Setup necessary directories for data and results"""
        base_dir = Path('data')
        maf_dir = base_dir / 'maf'
        results_dir = base_dir / 'results'
        
        for directory in [base_dir, maf_dir, results_dir]:
            directory.mkdir(exist_ok=True)
            
        return maf_dir, results_dir

    def read_maf(self, filepath: str) -> Optional[pd.DataFrame]:
        """Read MAF file with comprehensive error handling"""
        try:
            # Read the file and skip metadata lines starting with #
            df = pd.read_csv(filepath, sep='\t', comment='#', low_memory=False)
            
            # Basic validation
            required_cols = ['Hugo_Symbol', 'Chromosome', 'Start_Position', 'Variant_Type']
            missing = [col for col in required_cols if col not in df.columns]
            if missing:
                self.logger.warning(f"Missing required columns in {filepath}: {missing}")
            
            # Convert chromosome to string and ensure proper formatting
            df['Chromosome'] = df['Chromosome'].astype(str)
            
            # Handle missing values
            df = df.replace({'': np.nan, '.': np.nan})
            
            return df
            
        except Exception as e:
            self.logger.error(f"Error reading MAF file {filepath}: {str(e)}")
            return None

    def calculate_variant_metrics(self, df: pd.DataFrame) -> Dict:
        """Calculate comprehensive variant metrics"""
        metrics = {
            'variant_counts': {
                'total': len(df),
                'by_type': df['Variant_Type'].value_counts().to_dict(),
                'by_classification': df['Variant_Classification'].value_counts().to_dict(),
                'by_chromosome': df['Chromosome'].value_counts().to_dict()
            },
            'impact_distribution': df['IMPACT'].value_counts().to_dict() if 'IMPACT' in df.columns else {},
            'consequence_distribution': df['Consequence'].value_counts().to_dict() if 'Consequence' in df.columns else {},
            'clinical_significance': df['CLIN_SIG'].value_counts().to_dict() if 'CLIN_SIG' in df.columns else {}
        }
        
        # Calculate variant allele frequencies if coverage data is available
        if all(col in df.columns for col in ['t_depth', 't_alt_count']):
            df['VAF'] = df['t_alt_count'] / df['t_depth']
            metrics['vaf_statistics'] = {
                'mean': df['VAF'].mean(),
                'median': df['VAF'].median(),
                'std': df['VAF'].std()
            }
            
        return metrics

    def extract_gene_level_summary(self, df: pd.DataFrame) -> Dict:
        """Extract gene-level mutation summary"""
        gene_summary = {
            'most_mutated_genes': df['Hugo_Symbol'].value_counts().head(20).to_dict(),
            'genes_by_impact': {}
        }
        
        if 'IMPACT' in df.columns:
            impact_groups = df.groupby(['Hugo_Symbol', 'IMPACT']).size().unstack(fill_value=0)
            gene_summary['genes_by_impact'] = impact_groups.to_dict()
            
        return gene_summary

    def analyze_sample(self, filepath: str, results_dir: Path) -> Dict:
        """Analyze a single MAF file with comprehensive metrics"""
        df = self.read_maf(filepath)
        if df is None:
            return {}
            
        sample_name = Path(filepath).stem
        
        # Generate comprehensive analysis
        analysis = {
            'sample_name': sample_name,
            'file_stats': {
                'total_variants': len(df),
                'file_path': str(filepath),
                'creation_date': os.path.getctime(filepath)
            },
            'variant_metrics': self.calculate_variant_metrics(df),
            'gene_summary': self.extract_gene_level_summary(df)
        }
        
        # Save processed data
        processed_dir = results_dir / 'processed_data'
        processed_dir.mkdir(exist_ok=True)
        
        # Save different variant types separately
        for variant_type in df['Variant_Type'].unique():
            subset = df[df['Variant_Type'] == variant_type]
            subset.to_csv(processed_dir / f'{sample_name}_{variant_type.lower()}.csv', index=False)
        
        # Save complete processed data
        df.to_csv(processed_dir / f'{sample_name}_complete.csv', index=False)
        
        return analysis

    def process_all_samples(self) -> Dict:
        """Process all MAF files in the data directory"""
        maf_dir, results_dir = self.setup_directories()
        
        # Find all MAF files
        maf_files = glob.glob(str(maf_dir / '*.maf'))
        if not maf_files:
            self.logger.warning("No MAF files found in data/maf directory")
            return {}
            
        # Process each file
        results = {
            'sample_analyses': {},
            'comparative_analysis': {
                'total_samples': len(maf_files),
                'shared_variants': {},
                'unique_variants': {}
            }
        }
        
        # Analyze individual samples
        for maf_file in maf_files:
            sample_analysis = self.analyze_sample(maf_file, results_dir)
            results['sample_analyses'][Path(maf_file).stem] = sample_analysis
            
        # Save results
        with open(results_dir / 'complete_analysis.json', 'w') as f:
            json.dump(results, f, indent=4)
            
        self.logger.info(f"Analysis complete. Processed {len(maf_files)} MAF files.")
        return results

def main():
    analyzer = MAFAnalyzer()
    results = analyzer.process_all_samples()
    
    # Log summary statistics
    logging.info("Analysis Summary:")
    for sample, analysis in results['sample_analyses'].items():
        if analysis:  # Check if analysis exists
            variant_count = analysis.get('file_stats', {}).get('total_variants', 0)
            logging.info(f"{sample}: {variant_count} variants processed")

"""
RUN IN TERMINAL:
> python scripts/process-maf-data.py
2024-11-20 11:10:42,709 - INFO - Analysis complete. Processed 4 MAF files.
2024-11-20 11:10:42,709 - INFO - Analysis Summary:
2024-11-20 11:10:42,709 - INFO - MMCID-26B: 535 variants processed
2024-11-20 11:10:42,709 - INFO - MMCID-30B: 275 variants processed
2024-11-20 11:10:42,709 - INFO - TCMK1-14B: 327 variants processed
2024-11-20 11:10:42,709 - INFO - TMCK1-23B: 322 variants processed
"""

'\nRUN IN TERMINAL:\n> python scripts/process-maf-data.py\n2024-11-20 11:10:42,709 - INFO - Analysis complete. Processed 4 MAF files.\n2024-11-20 11:10:42,709 - INFO - Analysis Summary:\n2024-11-20 11:10:42,709 - INFO - MMCID-26B: 535 variants processed\n2024-11-20 11:10:42,709 - INFO - MMCID-30B: 275 variants processed\n2024-11-20 11:10:42,709 - INFO - TCMK1-14B: 327 variants processed\n2024-11-20 11:10:42,709 - INFO - TMCK1-23B: 322 variants processed\n'

In [1]:
import numpy as np
import pandas as pd
from typing import List, Tuple, Union, Dict
from pathlib import Path
from glob import glob
import logging
import json
import os, re
from pprint import pprint

root = Path(os.getcwd()).absolute()
data_path = root / 'data'
maf_path = data_path / 'maf'

In [2]:
# MAFParse
class MAFParse: 
    def __init__(self) -> None:
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

        self.crucial_columns = [
            'Hugo_Symbol',            # Gene name
            'Chromosome',             # Chromosome location
            'Start_Position',         # Start position
            'End_Position',           # End position
            'Variant_Classification', # Type of variant
            'Variant_Type',          # SNP, INS, DEL, etc.
            'Reference_Allele',      # Reference allele
            'Tumor_Seq_Allele1',     # Tumor allele 1
            'Tumor_Seq_Allele2',     # Tumor allele 2
            'dbSNP_RS',              # Known variant ID
            'HGVSp_Short',           # Protein change
            'SIFT',                  # SIFT prediction
            'PolyPhen',              # PolyPhen prediction
            'IMPACT',                # Variant impact
            'Consequence'            # Variant consequence
        ]

    def read_maf(self, filepath: str) -> pd.DataFrame:
        """Read and validate MAF file"""
        try:
            df = pd.read_csv(filepath, sep='\t', comment='#', low_memory=False)
            
            # Validate minimum required columns
            min_required = ['Hugo_Symbol', 'Chromosome', 'Start_Position', 'Variant_Classification']
            missing_cols = [col for col in min_required if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Missing required columns: {missing_cols}")
                
            self.logger.info(f"Successfully read MAF file: {filepath}")
            return df
            
        except Exception as e:
            self.logger.error(f"Error reading MAF file {filepath}: {str(e)}")
            raise
            
    def extract_crucial_cols(self, df:pd.DataFrame) -> pd.DataFrame:
        try:
            available_columns = [col for col in self.crucial_columns if col in df.columns]
            df_crucial = df[available_columns].copy()

            # Add derived columns
            df_crucial['Is_Novel'] = df['dbSNP_RS'].isna() # is/is not present in dbSNP database
            
            # clean specific columns 
            # check if 'HGVSp_Short' col else fill with 'Unknown'
            if 'HGVSp_Short' in df_crucial.columns:
                df_crucial['HGVSp_Short'] = df_crucial['HGVSp_Short'].fillna('Unknown')
                
            return df_crucial
        
        except Exception as e: # what is the best way to handle this?
            self.logger.error(f"Error extracting crucial info: {str(e)}")
            raise
            
    def process_file(self, filepath:str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """Process single MAF file and retunr both full/crucial dataframes"""
        try: 
            df_full = self.read_maf(filepath=filepath)
            df_crucial = self.extract_crucial_cols(df=df_full)
            return df_full, df_crucial
        except Exception as e:
            self.logger.error(f"Error processing MAF file: {str(e)}")
            raise


In [3]:
# MAFAnalyze
class MAFAnalyze:
    def __init__(self) -> None:
        self.logger = logging.getLogger(__name__)

    def _classify_clinical_significance(self, variant_info:pd.Series) -> str:
        """Classify variant clinical significance"""
        try:
            # check for high impact variants
            if variant_info['Variant_Classification'] in ['Nonsense_Mutation', 'Frame_Shift_Del', 'Frame_Shift_Ins', 'Splice_Site']:
                return 'High'
            
            # check low impact variants
            if variant_info['Variant_Classification'] in ['Silent', 'Intron', "3'UTR", "5'UTR"]:
                return 'Low'

        except Exception as e:
            return 'Unknown'
    
    def get_variant_summary(self, df:pd.DataFrame, filename:str='') -> Dict:
        """get comprehensive variant summary"""
        try: 
            df['Clinical_Significance'] = df.apply(self._classify_clinical_significance, axis=1)
            
            summary = {
                'filename': filename,
                'total_variants': len(df),
                'variant_types': df['Variant_Type'].value_counts().to_dict(),
                'variant_classifications': df['Variant_Classification'].value_counts().to_dict(),
                'chromosomal_distribution': df['Chromosome'].value_counts().to_dict(),
                'novel_variants': int(df['Is_Novel'].sum()),
                'known_variants': int((~df['Is_Novel']).sum()),
                'clinical_significance': df['Clinical_Significance'].value_counts().to_dict()
            }
            
            # Add high impact variants detail
            high_impact = df[df['Clinical_Significance'] == 'High']
            summary['high_impact_variants'] = {
                'count': len(high_impact),
                'genes': high_impact['Hugo_Symbol'].unique().tolist()
            }
            
            return summary
            
        except Exception as e:
            self.logger.error(f"Error generating variant summary: {str(e)}")
            raise

    def compare_samples(self, df1:pd.DataFrame, df2:pd.DataFrame,
                        name1:str = 'Sample1', name2:str = 'Sample2') -> Dict:
        """Compare two MAF samples"""
        try:
            def create_variant_id(df):
                # create id like: chr1_12345_A_T
                return df.apply(lambda x: f"{x['Chromosome']}_{x['Start_Position']}_"f"{x['Reference_Allele']}_{x['Tumor_Seq_Allele2']}", axis=1)
            
            df1, df2 = df1.copy(), df2.copy()
            df1['Variant_ID'] = create_variant_id(df1)
            df2['Variant_ID'] = create_variant_id(df2)
        
            shared_variants = set(df1['Variant_ID']).intersection(set(df2['Variant_ID']))
            unique_to_1 = set(df1['Variant_ID']).difference(set(df2['Variant_ID']))
            unique_to_2 = set(df2['Variant_ID']).difference(set(df1['Variant_ID']))

            # clinical significance if not present
            if 'Clinical_Significance' not in df1.columns:
                df1['Clinical_Significance'] = df1.apply(self._classify_clinical_significance, axis=1)
            if 'Clinical_Significance' not in df2.columns:
                df2['Clinical_Significance'] = df2.apply(self._classify_clinical_significance, axis=1)
            
            comparison = {
                'shared_variants': {
                    'count': len(shared_variants),
                    'variants': list(shared_variants)
                },
                f'unique_to_{name1}': {
                    'count': len(unique_to_1),
                    'high_impact_genes': df1[
                        (df1['Variant_ID'].isin(unique_to_1)) & 
                        (df1['Clinical_Significance'] == 'High')
                    ]['Hugo_Symbol'].unique().tolist()
                },
                f'unique_to_{name2}': {
                    'count': len(unique_to_2),
                    'high_impact_genes': df2[
                        (df2['Variant_ID'].isin(unique_to_2)) & 
                        (df2['Clinical_Significance'] == 'High')
                    ]['Hugo_Symbol'].unique().tolist()
                }
            }
            
            return comparison

        except Exception as e:
            self.logger.error(f"Error comparing samples: {str(e)}")
            raise

def find_maf_pairs(dir: Path) -> List[Tuple[str, str]]:
    """
    Find paired MAF files based on common prefixes (MMCID or TCMK1)
    """
    try:
        # Get all MAF files
        files = list(dir.glob('*.maf'))
        
        # Group files by their prefix
        mmcid_files = [f for f in files if 'MMCID' in f.name]
        tcmk1_files = [f for f in files if 'TCMK1' in f.name]
        
        pairs = []
        
        # Pair MMCID files
        if len(mmcid_files) == 2:
            pairs.append((str(mmcid_files[0]), str(mmcid_files[1])))
            
        # Pair TCMK1 files
        if len(tcmk1_files) == 2:
            pairs.append((str(tcmk1_files[0]), str(tcmk1_files[1])))
            
        logging.info(f"Found {len(pairs)} pairs: {pairs}")
        return pairs
        
    except Exception as e:
        logging.error(f"Error finding MAF pairs: {str(e)}")
        return []


def analyze_maf_dir(dir:str) -> Dict:
    """analyze all maf files"""
    parse = MAFParse()
    analyzer = MAFAnalyze()
    results = {}

    # find paired MAF files
    pairs = find_maf_pairs(dir=dir)
    for f1, f2 in pairs:
        try:
            # parse files
            _, df_crucial1 = parse.process_file(filepath=f1)
            _, df_crucial2 = parse.process_file(filepath=f2)
            
            # generate summaries
            summary1 = analyzer.get_variant_summary(df=df_crucial1, filename=f1)
            summary2 = analyzer.get_variant_summary(df=df_crucial2, filename=f2)

            # comparison
            comparison = analyzer.compare_samples(
                df1=df_crucial1, df2=df_crucial2,
                name1=Path(f1).stem, name2=Path(f2).stem
            )

            pair_name = f'{Path(f1).stem}_{Path(f2).stem}'
            results[pair_name] = {
                'sample_summary1': summary1,
                'sample_summary2': summary2,
                'comparison': comparison
            }
        except Exception as e:
            logging.error(f"Error processing pair {f1}, {f2}: {str(e)}")
            continue
        return results

def save_json(data: Dict, filename: str):
    """
    Save a dictionary to a JSON file with indentation.

    Parameters:
    data (Dict): The dictionary to save.
    filename (str): The name of the file to save the dictionary to.
    """
    with open(filename, 'w') as file:
        json.dump(data, file, indent=4)



parse = MAFParse()
analyze = MAFAnalyze()

sample1 = maf_path/'MMCID-26B.maf'
sample2 = maf_path/'MMCID-30B.maf'
parse = MAFParse()

df1_full, df1_crucial = parse.process_file(filepath=sample1)
summary = analyze.get_variant_summary(df=df1_crucial, filename=sample1)

df2_full, df2_crucial = parse.process_file(filepath=sample2)
summary = analyze.get_variant_summary(df=df2_crucial, filename=sample2)

# display(df1_full, df1_crucial, summary)
compare = analyze.compare_samples(df1=df1_crucial, df2=df2_crucial, name1='MMCID-26B', name2='MMCID-30B')
pprint(compare)

results = analyze_maf_dir(dir=maf_path)
pprint(results)

save_path = data_path / 'results'
save_json(data=results, filename=save_path/'results.json')
save_json(data=compare, filename=save_path/'compare.json')

INFO:__main__:Successfully read MAF file: /Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-26B.maf
INFO:__main__:Successfully read MAF file: /Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-30B.maf
INFO:root:Found 1 pairs: [('/Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-26B.maf', '/Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-30B.maf')]
INFO:__main__:Successfully read MAF file: /Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-26B.maf
INFO:__main__:Successfully read MAF file: /Users/adamkurth/Documents/vscode/research/ugenome/ugenome-maf-pdf/data/maf/MMCID-30B.maf


{'shared_variants': {'count': 75,
                     'variants': ['6_97145601_C_A',
                                  '1_170678304_G_A',
                                  '16_58455835_G_T',
                                  '7_28594252_T_C',
                                  '8_86091584_G_T',
                                  '11_113593899_G_T',
                                  '15_82501063_C_A',
                                  '9_107219941_C_-',
                                  '19_46272548_T_G',
                                  '7_28530518_C_-',
                                  '9_51760417_G_A',
                                  '8_69347314_C_T',
                                  '12_112727362_A_-',
                                  '1_88180391_C_T',
                                  '4_148024136_A_G',
                                  '5_96995348_CTCCTGCTCCTC_-',
                                  '6_90442452_G_T',
                                  '7_27021916_G_A',
         

---

report-generator.py
===================

1. HTML to tables download (of summary current output) in pdf report.
2. top 5 in clinical report 
3. put all in .csv file
4. for mutations for SNV (single nucleotide variant) and INDEL (insertion/deletion)
5. csv for SNV AND INDEL in summary documents.
6. download summary document.

PDF report:
============
Top section: Patient ID XXXXX
1. Genetic Characterization
2. SNV
3. INDEL


In [14]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.platypus import Flowable, Image
from reportlab.lib.enums import TA_LEFT, TA_CENTER
import datetime
import pandas as pd
import json
from pathlib import Path

class CSVLink(Flowable):
    def __init__(self, x, y, filename):
        Flowable.__init__(self)
        self.x = x
        self.y = y
        self.filename = filename
        
    def draw(self):
        self.canv.linkURL(
            f"file:{self.filename}",
            (self.x, self.y, self.x + 100, self.y + 10),
            relative=1
        )

class MAFClinicalReport:
    def __init__(self, output_path: str):
        self.doc = SimpleDocTemplate(
            output_path,
            pagesize=letter,
            rightMargin=30,
            leftMargin=30,
            topMargin=30,
            bottomMargin=30
        )
        self.styles = getSampleStyleSheet()
        self.elements = []
        
        # Custom styles
        self.styles.add(ParagraphStyle(
            'SectionHeader',
            parent=self.styles['Heading1'],
            fontSize=10,
            textColor=colors.HexColor('#4472C4'),
            spaceAfter=6
        ))

    def load_analysis_data(self, results_json: str, snv_csv: str, indel_csv: str):
        """Load analysis results and variant data"""
        with open(results_json) as f:
            self.analysis_data = json.load(f)
            
        self.snv_data = pd.read_csv(snv_csv)
        self.indel_data = pd.read_csv(indel_csv)

    def create_header(self):
        """Create report header"""
        current_date = datetime.datetime.now().strftime('%Y-%m-%d')
        report_num = f"MAF-{datetime.datetime.now().strftime('%Y%m%d')}"
        
        header_data = [
            ['Report N°', report_num],
            ['Date', current_date],
            ['ADDRESSEE'],
            ['• Ordering Center', '• Oncologist / Pathologist', '• Date of request'],
            ['_______________', '_______________', '_______________']
        ]
        
        style = TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#EDF3F7'))
        ])
        
        header = Table(header_data, colWidths=[2*inch, 2*inch, 2*inch])
        header.setStyle(style)
        self.elements.append(header)
        self.elements.append(Spacer(1, 0.1*inch))

    def create_patient_info(self):
        """Create patient information section"""
        patient_data = [
            ['PATIENT'],
            ['ID N°', 'XXXXX'],
            ['Clinical Diagnosis', 'mCRC'],
            ['Clinical Trial', '______________']
        ]
        
        style = TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#EDF3F7'))
        ])
        
        patient = Table(patient_data, colWidths=[1.5*inch, 4.5*inch])
        patient.setStyle(style)
        self.elements.append(patient)
        self.elements.append(Spacer(1, 0.1*inch))

    def create_assay_info(self):
        """Create assay information section"""
        summary = self.analysis_data['sample_summaries']['MMCID-26B']
        
        assay_data = [
            ['ASSAY'],
            ['Genomic Target', 'WES', 'Sequencer', 'HiSeq2000'],
            ['Target size', '33,000,000 bp', 'Run QC outcome:', 'Passed'],
            ['Total Variants', str(summary['total_variants'])],
            ['SNVs', str(summary['snv_count']), 'INDELs', str(summary['indel_count'])]
        ]
        
        style = TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#EDF3F7'))
        ])
        
        assay = Table(assay_data, colWidths=[1.25*inch]*4)
        assay.setStyle(style)
        self.elements.append(assay)
        self.elements.append(Spacer(1, 0.2*inch))

    def create_variant_tables(self):
        """Create SNV and INDEL tables with download links"""
        # SNV Table
        self.elements.append(Paragraph('SNVs identified', self.styles['SectionHeader']))
        
        snv_cols = ['Hugo_Symbol', 'Chromosome', 'Start_Position', 
                   'Reference_Allele', 'Tumor_Seq_Allele2', 'HGVSp_Short']
        snv_data = [snv_cols]  # Header row
        snv_data.extend(self.snv_data[snv_cols].head(10).values.tolist())
        
        style = TableStyle([
            ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('FONTSIZE', (0, 0), (-1, -1), 8),
            ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#EDF3F7')),
            ('GRID', (0, 0), (-1, -1), 0.25, colors.grey)
        ])
        
        snv_table = Table(snv_data, colWidths=[inch]*6)
        snv_table.setStyle(style)
        self.elements.append(snv_table)
        self.elements.append(CSVLink(0, 0, 'snvs.csv'))
        self.elements.append(Spacer(1, 0.2*inch))
        
        # INDEL Table
        self.elements.append(Paragraph('INDELs identified', self.styles['SectionHeader']))
        
        indel_cols = ['Hugo_Symbol', 'Chromosome', 'Start_Position', 
                     'Reference_Allele', 'Tumor_Seq_Allele2', 'HGVSp_Short']
        indel_data = [indel_cols]  # Header row
        indel_data.extend(self.indel_data[indel_cols].head(10).values.tolist())
        
        indel_table = Table(indel_data, colWidths=[inch]*6)
        indel_table.setStyle(style)
        self.elements.append(indel_table)
        self.elements.append(CSVLink(0, 0, 'indels.csv'))

    def generate_report(self):
        """Generate complete clinical report"""
        self.create_header()
        self.create_patient_info()
        self.create_assay_info()
        self.create_variant_tables()
        self.doc.build(self.elements)

def main():
    # Paths to input files
    results_json = 'data/results/analysis_results.json'
    snv_csv = 'analysis_output/MMCID-26B_snvs.csv'
    indel_csv = 'analysis_output/MMCID-26B_indels.csv'
    
    # Generate report
    report = MAFClinicalReport('clinical_report.pdf')
    report.load_analysis_data(results_json, snv_csv, indel_csv)
    report.generate_report()

if __name__ == "__main__":
    main()